In [6]:
from pathlib import Path
import importlib

import numpy as np
import solidspy.assemutil as ass

import utils.periodic_fem as periodic_fem

periodic_fem = importlib.reload(periodic_fem)

IMAGE_DATASET_PATH = Path("data/binary_image_dataset.npz")
RESULTS_PATH = Path("data/periodic_fem_results.npz")
MAX_IMAGES = 100
CELL_LENGTH = 1.0
BASE_DISPLACEMENT = 1.0
LOAD_MODES = ("tension_x", "tension_y", "shear")

materials = np.array(
    [
        [831e6, 0.35],
        [12e9, 0.27],
    ]
)

with np.load(IMAGE_DATASET_PATH) as dataset:
    images = dataset["images"]

if images.ndim != 3 or images.shape[0] == 0:
    raise ValueError("The image dataset must contain at least one 2D image.")

start_index = 0
end_index = min(start_index + MAX_IMAGES, images.shape[0])
image_indices = np.arange(start_index, end_index)

nodes = periodic_fem.create_node_grid(images[0], cell_length=CELL_LENGTH)
reference_nodes, image_nodes = periodic_fem.identify_periodic_nodes(nodes)
reference_dofs, reference_dof_map = periodic_fem.create_node_dof_map(reference_nodes)
image_dofs, image_dof_map = periodic_fem.create_node_dof_map(image_nodes)
total_dofs = 2 * len(nodes)
reduced_dofs = periodic_fem.reassign_periodic_dofs(
    total_dofs, image_dofs, reference_dofs
)

results = np.zeros((len(image_indices), 6))
for result_index, image_index in enumerate(image_indices):
    image = images[image_index]
    elements = periodic_fem.create_element_topology(image).astype(int)
    assembly_operator, boundary_conditions, equation_count = ass.DME(
        nodes[:, -2:], elements
    )
    stiffness_matrix, _ = ass.assembler(
        elements,
        materials,
        nodes[:, :3],
        equation_count,
        assembly_operator,
    )

    strain_cases = []
    stress_cases = []
    for mode in LOAD_MODES:
        transformation, prescribed = periodic_fem.build_periodic_transformation(
            BASE_DISPLACEMENT,
            total_dofs,
            reference_nodes,
            image_nodes,
            reference_dofs,
            image_dofs,
            reference_dof_map,
            image_dof_map,
            reduced_dofs,
            nodes=nodes,
            mode=mode,
        )
        strain_nodes, stress_nodes = periodic_fem.solve_periodic_load_case(
            stiffness_matrix,
            boundary_conditions,
            nodes,
            elements,
            materials,
            transformation,
            prescribed,
        )
        strain_cases.append(strain_nodes.T)
        stress_cases.append(stress_nodes.T)

    strain = np.concatenate(strain_cases, axis=1)
    stress = np.concatenate(stress_cases, axis=1)
    strain[2, :] *= 0.5
    stiffness = (stress @ strain.T) @ np.linalg.inv(strain @ strain.T)
    stiffness = 0.5 * (stiffness + stiffness.T)
    results[result_index] = np.array(
        [
            stiffness[0, 0],
            stiffness[1, 1],
            stiffness[2, 2],
            stiffness[0, 1],
            stiffness[1, 2],
            stiffness[0, 2],
        ]
    )

RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)
np.savez_compressed(
    RESULTS_PATH,
    results=results,
    image_indices=image_indices,
)
print(f"Processed images: {len(image_indices)}")
print(f"Results file: {RESULTS_PATH}")

Processed images: 100
Results file: data/periodic_fem_results.npz
